In [9]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'

In [10]:
import torch
import datasets
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

In [11]:
base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

In [12]:
class Holder:
    pass

args = Holder()
args.n_layer = 4
args.n_head = 4
args.n_embd = 128

In [13]:
from torch.nn.utils.rnn import pad_sequence

In [14]:
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    segments_batch = []
    for sample in batch:
        context = sample['context']
        
        perform_memory_task = torch.rand(1) < args.memory_task_freq
        if perform_memory_task and args.memory_task == "reconstruct":
            query = '!?'
            target = '!?' + context[2:-2]
        elif perform_memory_task and args.memory_task == "continue":
            # print(f"[collate_fn] ", args.memory_key_size, args.memory_value_size, len(context))
            query_start_ind = torch.randint(0, len(context) - args.memory_key_size - args.memory_value_size - 4, (1,))
            query = '!?' + context[query_start_ind:query_start_ind + args.memory_key_size]
            target = '!?' + context[query_start_ind + args.memory_key_size:query_start_ind + args.memory_key_size + args.memory_value_size]
        else:
            query = sample['query']
            target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

In [15]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)['train']

In [16]:
args.memory_task = None
args.memory_task_freq = 0.5

In [17]:
batch = [dataset[i] for i in range(10)]
collated = collate_fn(batch)


In [18]:
from transformers import AutoConfig
from transformers import AutoModelForCausalLM

base_model_config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
base_model_config.num_hidden_layers = args.n_layer
base_model_config.num_attention_heads = args.n_head
base_model_config.num_key_value_heads = args.n_head
base_model_config.hidden_size = args.n_embd
base_model_config.head_dim = base_model_config.hidden_size // base_model_config.num_attention_heads
base_model_config.intermediate_size = base_model_config.hidden_size * 4

In [21]:
# from modeling_rmt.huggingface import RMTForReasoning, RMTConfig
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/modeling_rmt")
from huggingface import *


In [57]:
class RMTVLForReasoning(PreTrainedModel):
    config_class = RMTConfig

    def __init__(self, config: RMTConfig, **kwargs):
        super().__init__(config, **kwargs)
        from transformers import AutoConfig, AutoModelForCausalLM
        if config.from_pretrained:
            base_model = AutoModelForCausalLM.from_pretrained(config.from_pretrained)
        else:
            if config.base_model_config is None:
                base_config = AutoConfig.from_pretrained(config.base_model_name)
            else:
                base_config = config.base_model_config
            base_model = AutoModelForCausalLM.from_config(base_config)

        self.rmt_config = config
        memory_cell = MemoryCellVariableLayer(base_model, num_mem_tokens=config.num_mem_tokens, out_layer_idx=config.out_layer_idx)
        self.rmt = RecurrentWrapperNoSegmentationGenerate(
            memory_cell,
            max_n_segments=config.max_n_segments,
            think_token_id=config.think_token_id,
            answer_token_id=config.answer_token_id,
            bos_token_id=config.bos_token_id,
            eos_token_id=config.eos_token_id
        )

    def forward(self, labels=None, *args, **kwargs):
        return self.rmt(labels=labels, *args, **kwargs)

    def generate(self, *args, **kwargs):
        return self.rmt.generate(*args, **kwargs)

    def load_state_dict(self, state_dict, strict=True, assign=False):
        try:
            return super().load_state_dict(state_dict, strict, assign)
        except RuntimeError:
            print("Failed to load state, retrying with RMT loader.")
            self.rmt.load_state_dict(state_dict, strict=True, assign=assign)
            print("Success!")

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, config=None, *args, **kwargs):
        from transformers.utils.hub import cached_file, HfHubHTTPError
        import torch

        if config is None:
            config = RMTConfig.from_pretrained(pretrained_model_name_or_path, **kwargs)

        model = cls(config)

        state_dict = None
        try:
            weights_path = cached_file(pretrained_model_name_or_path, "model.safetensors", **kwargs)
            from safetensors.torch import load_file
            state_dict = load_file(weights_path, device="cpu")
        except (OSError, HfHubHTTPError):
            try:
                weights_path = cached_file(pretrained_model_name_or_path, "pytorch_model.bin", **kwargs)
                state_dict = torch.load(weights_path, map_location="cpu")
            except (OSError, HfHubHTTPError):
                print(f"Warning: Could not find weights for {pretrained_model_name_or_path}. "
                      f"The model is initialized randomly.")

        if state_dict is not None:
            model.load_state_dict(state_dict, strict=False)

        return model


class MemoryCellVariableLayer(torch.nn.Module):
    def __init__(self, base_model, num_mem_tokens, out_layer_idx=-1):
        super().__init__()
        self.model = base_model
        self.out_layer_idx = out_layer_idx
        self.create_memory(num_mem_tokens)

    def create_memory(self, num_mem_tokens):
        self.num_mem_tokens = num_mem_tokens
        embeddings = self.model.get_input_embeddings()
        memory_dim = getattr(self.model.config, 'n_embd', self.model.config.hidden_size)
        memory_weights = torch.randn((num_mem_tokens, memory_dim)) * embeddings.weight.data.std()
        self.register_parameter('memory', torch.nn.Parameter(memory_weights, requires_grad=True))

        self.read_memory_position = range(num_mem_tokens)
        self.write_memory_position = range(-num_mem_tokens, 0)

    def set_memory(self, input_shape):
        memory = self.memory.repeat(input_shape[0], 1, 1)
        return memory

    def forward(self, input_ids, memory_state=None, **kwargs):
        if memory_state is None:
            memory_state = self.set_memory(input_ids.shape)

        seg_kwargs = self.process_input(input_ids, memory_state, write_mem=True, **kwargs)
        out = self.model(**seg_kwargs)
        out, new_memory_state = self.process_output(out, **kwargs)

        return out, new_memory_state

    def generate(self, input_ids, memory_state, attention_mask=None, **generate_kwargs):
        if memory_state is None:
            memory_state = self.set_memory(input_ids.shape)

        seg_kwargs = self.process_input(input_ids, memory_state, attention_mask=attention_mask, write_mem=False)
        out = self.model.generate(inputs_embeds=seg_kwargs['inputs_embeds'],
                                  attention_mask=seg_kwargs['attention_mask'],
                                  **generate_kwargs)
        return out

    def process_input(self, input_ids, memory_state, write_mem, **kwargs):
        seg_kwargs = dict(**kwargs)

        inputs_embeds = kwargs.get('inputs_embeds')
        if inputs_embeds is None:
            inputs_embeds = self.model.get_input_embeddings()(input_ids)

        if self.num_mem_tokens > 0:
            if write_mem:
                inputs_embeds = torch.cat([memory_state, inputs_embeds, memory_state], dim=1)
            else:
                inputs_embeds = torch.cat([memory_state, inputs_embeds], dim=1)

        seg_kwargs['input_ids'] = None
        seg_kwargs['inputs_embeds'] = inputs_embeds
        if kwargs.get('attention_mask') is not None:
            seg_kwargs['attention_mask'] = self.pad_attention_mask(kwargs['attention_mask'], inputs_embeds.shape)
        seg_kwargs['output_hidden_states'] = True
        return seg_kwargs

    def pad_attention_mask(self, attention_mask, shape):
        if self.num_mem_tokens in {0, None}:
            return attention_mask
        else:
            mask = torch.ones(*shape[:2], dtype=torch.int64).to(attention_mask.device)
            mask[:, self.num_mem_tokens: self.num_mem_tokens + attention_mask.shape[1]] = attention_mask
            return mask

    def process_output(self, model_outputs, **kwargs):
        if self.num_mem_tokens not in {0, None}:
            out = CausalLMOutputWithCrossAttentions()
            memory_state = model_outputs.hidden_states[self.out_layer_idx][:, -self.num_mem_tokens:]
            out['logits'] = model_outputs.logits[:, self.num_mem_tokens:-self.num_mem_tokens]

            if kwargs.get('output_hidden_states'):
                out['hidden_states'] = [lh[:, self.num_mem_tokens:-self.num_mem_tokens]
                                        for lh in model_outputs.hidden_states]
            if kwargs.get('output_attentions'):
                out['attentions'] = model_outputs['attentions']
        else:
            memory_state = None
            out = model_outputs

        return out, memory_state

In [58]:
# class RecurrentWrapperNoSegmentationGenerate(RecurrentWrapperNoSegmentation):
#     def forward(self, segments, labels, output_attentions=None, output_hidden_states=None, *args, **kwargs):
#         memory_state = None

#         cell_outputs = []
#         for seg_num, segment in enumerate(segments):
#             cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
#                                                       attention_mask=segment['attention_mask'],
#                                                       memory_state=memory_state, output_hidden_states=True)
#             cell_outputs.append(cell_out)
#             self.manage_gradients(memory_state, seg_num)

#         out = self.process_outputs(cell_outputs, segments,
#                                    output_attentions=output_attentions,
#                                    output_hidden_states=output_hidden_states)
#         return out

#     def generate(self, segments, **kwargs):
#         memory_state = None

#         for seg_num, segment in enumerate(segments):
#             cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
#                                                       attention_mask=segment['attention_mask'],
#                                                       memory_state=memory_state, output_hidden_states=True)

#         generated_segments = []
#         for seg_num in range(len(segments), self.rmt_config.get("max_n_segments", 32)):
#             output_ids, memory_state = self.generate_segment(memory_state=memory_state, **kwargs)
#             generated_segments.append(output_ids)

#             if self.all_done(generated_segments):
#                 break

#         return generated_segments

#     def generate_segment(self, memory_state, **kwargs):
#         input_ids = self.get_bos_tensor(memory_state)
#         attention_mask = torch.ones_like(input_ids).bool()

#         generated = self.memory_cell.generate(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             memory_state=memory_state,
#             stopping_criteria=self.make_custom_stopping_criteria(),
#             **kwargs
#         )

#         # Update memory state from generation
#         fwd_inputs = torch.cat((input_ids, generated), dim=1)[:, :-1]
#         _, memory_state = self.memory_cell(input_ids=fwd_inputs, memory_state=memory_state)

#         return generated, memory_state

#     def get_bos_tensor(self, memory_state):
#         bos = self.rmt_config["bos_token_id"]
#         bos_tensor = torch.tensor([bos] * memory_state.shape[0]).reshape(-1, 1)
#         return bos_tensor.to(memory_state.device)

#     def all_done(self, generated_segments):
#         eos = self.rmt_config['eos_token_id']
#         bs = generated_segments[0].shape[0]
#         have_eos = [any([eos in seg[i] for seg in generated_segments]) for i in range(bs)]
#         all_done = all(have_eos)
#         return all_done

#     def make_custom_stopping_criteria(self):
#         return [StopOnSpecialTokenCriteria([self.rmt_config['think_token_id'], self.rmt_config['answer_token_id']])]


In [68]:
config = RMTConfig()
# config.base_model_name = "HuggingFaceTB/SmolLM2-135M"
config.base_model_config = base_model_config
config.num_mem_tokens = 16
config.max_n_segments = 10
config.think_token_id = 100
config.answer_token_id = 101
config.bos_token_id = 102
config.eos_token_id = 103

config.out_layer_idx = -2

model = RMTVLForReasoning(config)

# model.load_state_dict(torch.load("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/models/N8-K2V2-V62_1M/model.pt"))

In [69]:
device = torch.device('cuda')
collated['segments'] = [{k:v.to(device) for k,v in s.items()} for s in collated['segments']]
collated['labels'] = collated['labels'].to(device)
model.eval()
model.to(torch.bfloat16)
model.to(device)
':)'

':)'

In [70]:
def forward(self, segments, labels, output_attentions=None, output_hidden_states=None, *args, **kwargs):
        memory_state = None

        cell_outputs = []
        for seg_num, segment in enumerate(segments):
            cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                      attention_mask=segment['attention_mask'],
                                                      memory_state=memory_state, output_hidden_states=True)
            cell_outputs.append(cell_out)
            self.manage_gradients(memory_state, seg_num)

        out = self.process_outputs(cell_outputs, segments,
                                   output_attentions=output_attentions,
                                   output_hidden_states=output_hidden_states)
        return out


In [71]:
wrapper_out = forward(model.rmt, **collated)

In [72]:
self = model.rmt
segments = collated['segments']
output_attentions = False
output_hidden_states = True

In [73]:
memory_state = None

cell_outputs = []
for seg_num, segment in enumerate(segments):
    cell_out, memory_state = self.memory_cell(input_ids=segment['input_ids'],
                                                attention_mask=segment['attention_mask'],
                                                memory_state=memory_state, output_hidden_states=True)
    cell_outputs.append(cell_out)
    self.manage_gradients(memory_state, seg_num)

out = self.process_outputs(cell_outputs, segments,
                            output_attentions=output_attentions,
                            output_hidden_states=output_hidden_states)

In [78]:
memory_state.shape

torch.Size([10, 16, 128])

In [80]:
cell_out.hidden_states[-2].shape

torch.Size([10, 8, 128])

In [47]:
len(cell_out.hidden_states)

5

In [32]:
with torch.no_grad():
    out = model(**collated)

In [33]:
out.logits.shape

torch.Size([10, 57, 128256])

In [33]:
out.loss

tensor(5.9062, device='cuda:0', dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [34]:
tokenizer.batch_decode(collated['segments'][0]['input_ids'])

['!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|<|endoftext|><|endoftext|>',
 '!wO:em!!nB:zb!!Mq:tS!!Bi:Nf!!Bp:Zp!!wM:nt!!MP:Sj!!5o:iR!|<|endoftext|><|endoftext|><|endoftext|>',
 '!aO:Kx!!yA:62!!rO:iS!!Wi:1l!!GJ:ni!!po:DD!!43:zk!!C6:6i!|<|endoftext|><|endoftext|>',
 '!SJ:Wk!!LP:3D!!QE:yq!!Ea:Gd!!Ne:ef!!u4:ix!!vF:kx!!pZ:hN!|<|endoftext|><|endoftext|><|endoftext|>',
 '!qy:lx!!Nb:rK!!0D:Oa!!7f:Vr!!zZ:x7!!zN:q3!!IL:nK!!7j:3Z!|',
 '!qK:xl!!hE:lJ!!P9:Qg!!o6:DG!!KW:6w!!Lz:BW!!Dj:xl!!gn:4o!|<|endoftext|><|endoftext|>',
 '!zV:Tk!!MT:7A!!D2:k7!!7G:91!!le:2s!!ce:9g!!Re:Bu!!qz:fr!|<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>',
 '!aa:Bt!!SF:qp!!U1:OF!!6x:cr!!kV:5B!!Qm:WC!!Zu:J0!!g5:2R!|<|endoftext|><|endoftext|><|endoftext|><|endoftext|>',
 '!fm:Yk!!eJ:Xt!!0V:WI!!pd:q3!!MT:rY!!XC:3U!!Q0:kC!!fh:Gr!|<|endoftext|><|endoftext|><|endoftext|><|endoftext|>',
 '!t4:ur!!TU:fI!!WF:Hx!!ZM:PB!!lW:B5!!L8:cE!!WG:4R!!wl:qp!|<|endoftext|><|endoftext|>']

In [ ]:
tokenizer.batch_decode(collated['segments'][1]['input_ids'])

['? ! I 9 : d t ! |',
 '? ! w M : n t ! |',
 '? ! a O : K x ! |',
 '? ! L P : 3 D ! |',
 '? ! 7 f : V r ! |',
 '? ! P 9 : Q g ! |',
 '? ! D 2 : k 7 ! |',
 '? ! Q m : W C ! |',
 '? ! 0 V : W I ! |',
 '? ! T U : f I ! |']

In [ ]:
for l, m in zip(collated['segments'][1]['input_ids'], collated['segments'][1]['labels_mask']):
    print(tokenizer.decode(l[m]))


: d t ! |
: n t ! |
: K x ! |
: 3 D ! |
: V r ! |
: Q g ! |
: k 7 ! |
: W C ! |
: W I ! |
: f I ! |


In [ ]:
memory_cell = MemoryCell(config)

In [23]:
collated['input_ids'].shape

torch.Size([10, 2, 49])

In [24]:
out.loss

tensor(8.6703, device='cuda:0', grad_fn=<DivBackward0>)

In [1]:
torch.ones(10, 10)

NameError: name 'torch' is not defined

In [26]:
out.keys()

odict_keys(['loss', 'ce_loss', 'logits', 'logits_0', 'ce_loss_0', 'logits_1', 'ce_loss_1'])

In [25]:
out.logits.shape

torch.Size([10, 98, 128256])

RMT

In [ ]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)

In [3]:
ds = dataset['train']

In [4]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/tokenizers/kv_alphabet_62")

In [6]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_rmt.huggingface import RMTForReasoning, RMTConfig


[2025-09-01 11:56:57,732] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-01 11:56:59,903] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [29]:
from transformers import AutoConfig, AutoModelForCausalLM
cfg_name = "HuggingFaceTB/SmolLM2-360M"
model_cfg = AutoConfig.from_pretrained(cfg_name)
base_model = AutoModelForCausalLM.from_config(model_cfg, use_flash_attn=True)


TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'use_flash_attn'

In [24]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (rotary_emb): LlamaRotaryEm

In [7]:
class Holder:
    pass

args = Holder()
args.n_layer = 4
args.n_head = 4
args.n_embd = 128


In [8]:
from transformers import AutoConfig
from transformers import AutoModelForCausalLM

base_model_config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
base_model_config.num_hidden_layers = args.n_layer
base_model_config.num_attention_heads = args.n_head
base_model_config.num_key_value_heads = args.n_head
base_model_config.hidden_size = args.n_embd
base_model_config.head_dim = base_model_config.hidden_size // base_model_config.num_attention_heads
base_model_config.intermediate_size = base_model_config.hidden_size * 4

In [9]:

# base_model = AutoModelForCausalLM.from_config(config)

In [10]:
config = RMTConfig()
# config.base_model_name = "HuggingFaceTB/SmolLM2-135M"
config.base_model_config = base_model_config
config.num_mem_tokens = 16
config.max_n_segments = 10
config.think_token_id = 100
config.answer_token_id = 101
config.bos_token_id = 102
config.eos_token_id = 103

model = RMTForReasoning(config)

# model.load_state_dict(torch.load("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/models/N8-K2V2-V62_1M/model.pt"))

In [11]:
from modeling_rmt.language_modeling import MemoryCell, RecurrentWrapper

In [12]:
model.main_input_name

'input_ids'

In [13]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [14]:
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    segments_batch = []
    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

In [ ]:
batch = [ds[i] for i in range(10)]
collated = collate_fn(batch)


In [16]:
import torch

In [17]:
model.to(dtype=torch.bfloat16)
out = model(**collated)

In [18]:
collated['segments'][0]['input_ids'].shape, collated['segments'][1]['input_ids'].shape

(torch.Size([10, 57]), torch.Size([10, 9]))

In [19]:
out.logits.shape

torch.Size([10, 66, 128256])

In [20]:
out.loss

tensor(6., dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [21]:
tokenizer.batch_decode(collated['segments'][0]['input_ids'])

['! V 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t ! |',
 '! w O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R ! |',
 '! a O : K x ! ! y A : 6 2 ! ! r O : i S ! ! W i : 1 l ! ! G J : n i ! ! p o : D D ! ! 4 3 : z k ! ! C 6 : 6 i ! |',
 '! S J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N ! |',
 '! q y : l x ! ! N b : r K ! ! 0 D : O a ! ! 7 f : V r ! ! z Z : x 7 ! ! z N : q 3 ! ! I L : n K ! ! 7 j : 3 Z ! |',
 '! q K : x l ! ! h E : l J ! ! P 9 : Q g ! ! o 6 : D G ! ! K W : 6 w ! ! L z : B W ! ! D j : x l ! ! g n : 4 o ! |',
 '! z V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r ! |',
 '! a a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R ! |',
 '! f m : Y k ! ! e J : X t ! ! 0 V : W I ! ! p d : q 3 

In [22]:
tokenizer.batch_decode(collated['segments'][1]['input_ids'])

['? ! I 9 : d t ! |',
 '? ! w M : n t ! |',
 '? ! a O : K x ! |',
 '? ! L P : 3 D ! |',
 '? ! 7 f : V r ! |',
 '? ! P 9 : Q g ! |',
 '? ! D 2 : k 7 ! |',
 '? ! Q m : W C ! |',
 '? ! 0 V : W I ! |',
 '? ! T U : f I ! |']

In [33]:
for l, m in zip(collated['segments'][1]['input_ids'], collated['segments'][1]['labels_mask']):
    print(tokenizer.decode(l[m]))


: d t ! |
: n t ! |
: K x ! |
: 3 D ! |
: V r ! |
: Q g ! |
: k 7 ! |
: W C ! |
: W I ! |
: f I ! |


In [ ]:
memory_cell = MemoryCell(config)